# Image-to-3D Benchmark — BATCH 2: SPAR3D + TRELLIS
Evaluates **SPAR3D** and **TRELLIS** on the curated GSO benchmark.

In [ ]:
# --- Error logging setup ---
import sys, traceback, datetime
from pathlib import Path
LOG_FILE = Path("/kaggle/working/outputs/benchmark_log.txt")
LOG_FILE.parent.mkdir(parents=True, exist_ok=True)
class Tee:
    def __init__(self, *files):
        self.files = files
    def write(self, data):
        for f in self.files:
            f.write(data)
            f.flush()
    def flush(self):
        for f in self.files:
            f.flush()
log_f = open(LOG_FILE, "a")
sys.stdout = Tee(sys.stdout, log_f)
sys.stderr = Tee(sys.stderr, log_f)
print(f"=== BENCHMARK LOG START {datetime.datetime.now()} ===")

In [ ]:
import os, subprocess, sys, glob, gc, json
from pathlib import Path

# 1. Headless OpenGL & T4 CUDA compilation flags (sm_75 only -> 10x faster C++/CUDA builds!)
os.environ["PYOPENGL_PLATFORM"] = "egl"
os.environ["ATTN_BACKEND"] = "xformers"
os.environ["SPCONV_ALGO"] = "native"
os.environ["SPCONV_DISABLE_JIT"] = "1"
os.environ["TORCH_CUDA_ARCH_LIST"] = "7.5"
os.environ["MAX_JOBS"] = "4"

# 2. HuggingFace token check
_hf_token = None
try:
    for _k in ["HF_TOKEN", "hf_token", "HUGGINGFACE_TOKEN", "HUGGING_FACE_HUB_TOKEN"]:
        if os.environ.get(_k):
            _hf_token = os.environ[_k]
            print(f"HF_TOKEN found in environment via {_k} ✅")
            break
    if not _hf_token:
        from kaggle_secrets import UserSecretsClient
        _client = UserSecretsClient()
        for _k in ["HF_TOKEN", "hf_token", "HUGGINGFACE_TOKEN", "HUGGING_FACE_HUB_TOKEN"]:
            try:
                _val = _client.get_secret(_k)
                if _val:
                    _hf_token = _val
                    print(f"HF_TOKEN found in Kaggle Secrets under '{_k}' ✅")
                    break
            except Exception:
                pass
except Exception as _e:
    print(f"Notice: Kaggle secrets check: {type(_e).__name__}: {_e}")

if _hf_token:
    os.environ["HF_TOKEN"] = _hf_token
    os.environ["HUGGING_FACE_HUB_TOKEN"] = _hf_token
    os.environ["HUGGINGFACE_TOKEN"] = _hf_token
    try:
        from huggingface_hub import login
        login(token=_hf_token, write_to_disk=True)
    except Exception:
        pass
    print("HF_TOKEN configured ✅")
else:
    print("Notice: HF_TOKEN not attached (public/mirror models will still work).")

import torch
print(f"torch {torch.__version__} (CUDA: {torch.version.cuda})")
if not torch.cuda.is_available():
    raise RuntimeError("No CUDA GPU attached! Enable Accelerator > GPU T4 x2.")
props = torch.cuda.get_device_properties(0)
print(f"CUDA OK: {props.name} | {props.total_memory/1e9:.1f} GB VRAM")

NUM_SAMPLES = 10
GSO_DIR = "/kaggle/working/data/gso"
OUT = Path("/kaggle/working/outputs"); OUT.mkdir(parents=True, exist_ok=True)

import numpy
open("/tmp/constraints.txt", "w").write(f"numpy=={numpy.__version__}\n")
print(f"Config: NUM_SAMPLES={NUM_SAMPLES} | numpy pinned: {numpy.__version__}")

In [ ]:
import os, sys, subprocess, datetime
import numpy as _np
if not os.path.exists("/tmp/constraints.txt"):
    open("/tmp/constraints.txt", "w").write(f"numpy=={_np.__version__}\n")

!pip install -q -c /tmp/constraints.txt trimesh scipy lpips rembg onnxruntime pooch opencv-python-headless pyrender pandas huggingface_hub "Pillow>=10.0"

import torch
assert torch.cuda.is_available(), "torch lost CUDA during installs"
r = subprocess.run([sys.executable, "-c", "import rembg"], capture_output=True, text=True)
if r.returncode != 0:
    !pip install -q -c /tmp/constraints.txt --force-reinstall --no-deps onnxruntime

subprocess.run([sys.executable, "-c", "import lpips; lpips.LPIPS(net='vgg')"], check=True)
from rembg import new_session
_sess = new_session("isnet-general-use")
print("=== CELL 3 GATE: PASS (LPIPS + ISNet cached) ✅ ===")

In [ ]:
import os, sys, shutil, hashlib, subprocess

CODE_DIR = "/kaggle/working/image-to-3d-benchmark"
shutil.rmtree(CODE_DIR, ignore_errors=True)
_r = subprocess.run(
    ["git", "clone", "--depth", "1", "-b", "main", "https://github.com/anubhavaanand/image-to-3d.git", CODE_DIR],
    capture_output=True, text=True
)
if _r.returncode != 0 or not os.path.exists(os.path.join(CODE_DIR, "src")):
    raise RuntimeError("GitHub clone failed:\n" + _r.stderr[-500:])

_sha = subprocess.run(["git", "-C", CODE_DIR, "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip()
if CODE_DIR not in sys.path:
    sys.path.insert(0, CODE_DIR)

from src.evaluation.benchmark_runner import BenchmarkRunner, load_gso_samples, bake_texture_to_vertices
from src.engines.triposr_engine import TriposrEngine
from src.engines.lgm_engine import LGMEngine
from src.engines.trellis_engine import TrellisEngine
from src.engines.spar3d_engine import Spar3dEngine
print(f"code cloned fresh @ {_sha[:12]} — all 4 engines imported OK ✅")

In [ ]:
import os, sys, glob, zipfile, tempfile, shutil, random
from pathlib import Path
import numpy as np, trimesh
from PIL import Image as PILImage

os.environ.setdefault("PYOPENGL_PLATFORM", "egl")
if "GSO_DIR" not in globals():
    GSO_DIR = "/kaggle/working/data/gso"
if "NUM_SAMPLES" not in globals():
    NUM_SAMPLES = 10
if "CODE_DIR" not in globals():
    CODE_DIR = "/kaggle/working/image-to-3d-benchmark"
if str(CODE_DIR) not in sys.path:
    sys.path.insert(0, str(CODE_DIR))

from src.evaluation.benchmark_runner import load_gso_samples, bake_texture_to_vertices

CANONICAL_10_GSO = [
    "3D_Dollhouse_Happy_Brother",
    "3D_Dollhouse_Sofa",
    "ASICS_GELAce_Pro_Pearl_WhitePink",
    "ASICS_GELLinksmaster_WhiteCoffeeSand",
    "ASICS_GELResolution_5_Flash_YellowBlackSilver",
    "ASICS_HyperRocketgirl_SP_5_WhiteMalibu_BlueBlack",
    "Android_Figure_Panda",
    "Animal_Planet_Foam_2Headed_Dragon",
    "Azure_Snake_Tieks_Leather_Snake_Print_Ballet_Flats",
    "Bradshaw_International_11642_7_Qt_MP_Plastic_Bowl",
]

def count_valid(root):
    root = Path(root)
    if not root.exists():
        return 0
    return sum(
        1 for d in root.iterdir()
        if d.is_dir() and (list((d/"meshes").glob("*.obj")) or list((d/"meshes").glob("*.ply")))
    )

REPO = "suvadityamuk/google-scanned-objects-raw"
if count_valid(GSO_DIR) < NUM_SAMPLES:
    from huggingface_hub import HfApi, hf_hub_download
    api = HfApi()
    all_zips = sorted(f for f in api.list_repo_files(REPO, repo_type="dataset") if f.endswith(".zip"))
    if NUM_SAMPLES <= 10:
        wanted = [name + ".zip" for name in CANONICAL_10_GSO if (name + ".zip") in set(all_zips)]
    else:
        wanted = None
        for hp in ["/kaggle/working/handpicked.txt", os.path.join(CODE_DIR, "handpicked.txt")] + glob.glob("/kaggle/input/*/handpicked.txt"):
            if os.path.exists(hp):
                wanted = [l.strip() + ".zip" for l in open(hp) if l.strip() and not l.strip().startswith("#")]
                wanted = [w for w in wanted if w in set(all_zips)]
                break
        if wanted is None:
            random.Random(42).shuffle(all_zips)
            wanted = all_zips

    def ingest(zip_name):
        obj_id = zip_name[:-4]
        try:
            zp = hf_hub_download(REPO, zip_name, repo_type="dataset", local_dir="/kaggle/working/gso_zips")
            with tempfile.TemporaryDirectory() as td:
                with zipfile.ZipFile(zp) as z:
                    z.extractall(td)
                hits = list(Path(td).rglob("*.obj")) + list(Path(td).rglob("*.ply"))
                if not hits:
                    return False
                mdir = Path(GSO_DIR) / obj_id / "meshes"
                mdir.mkdir(parents=True, exist_ok=True)
                for h in hits:
                    shutil.copy(h, mdir / h.name)
                for sib in hits[0].parent.iterdir():
                    if sib.suffix.lower() in (".mtl", ".png", ".jpg", ".jpeg"):
                        shutil.copy(sib, mdir / sib.name)
                for tex in Path(td).rglob("*.png"):
                    if not (mdir / tex.name).exists():
                        shutil.copy(tex, mdir / tex.name)
            return True
        except Exception as e:
            print("  skip", obj_id, str(e)[:80])
            return False

    ok = 0
    target_n = 10 if NUM_SAMPLES <= 10 else min(len(wanted), NUM_SAMPLES + 10)
    for zname in wanted:
        if ingest(zname):
            ok += 1
        if ok >= target_n:
            break
    print(f"ingested {ok} objects into {GSO_DIR}")

def render_cam(tm, az, el, res, return_rgba=False):
    import pyrender
    try:
        tm.fix_normals()
    except Exception:
        pass
    # Balanced studio lighting (0.55 ambient + 1.0 key) so white shoes keep 3D shading and never blow out to 255!
    scene = pyrender.Scene(ambient_light=np.array([0.55, 0.55, 0.55]), bg_color=np.array([1.0, 1.0, 1.0, 1.0]))
    scene.add(pyrender.Mesh.from_trimesh(tm, smooth=False))
    azr, elr, dist = np.radians(az), np.radians(el), 2.0
    cam_pos = np.array([dist*np.cos(elr)*np.sin(azr), dist*np.sin(elr), dist*np.cos(elr)*np.cos(azr)], dtype=np.float32)
    forward = -cam_pos / (np.linalg.norm(cam_pos) + 1e-8)
    right = np.cross(forward, np.array([0.0, 1.0, 0.0], dtype=np.float32))
    right /= (np.linalg.norm(right) + 1e-8)
    up = np.cross(right, forward)
    pose = np.eye(4, dtype=np.float32)
    pose[:3, 0], pose[:3, 1], pose[:3, 2], pose[:3, 3] = right, up, -forward, cam_pos
    scene.add(pyrender.PerspectiveCamera(yfov=np.radians(45.0)), pose=pose)
    scene.add(pyrender.DirectionalLight(color=np.ones(3), intensity=1.0), pose=pose)
    ren = pyrender.OffscreenRenderer(res * 2, res * 2)
    col, depth = ren.render(scene, flags=pyrender.RenderFlags.SKIP_CULL_FACES)
    ren.delete()
    if return_rgba:
        alpha = ((depth > 0).astype(np.uint8) * 255)[..., None]
        rgba_hi = np.concatenate([col, alpha], axis=-1)
        return np.array(PILImage.fromarray(rgba_hi, mode="RGBA").resize((res, res), PILImage.LANCZOS))
    return np.array(PILImage.fromarray(col).resize((res, res), PILImage.LANCZOS))

rx_gso = np.array([[1, 0, 0, 0], [0, 0, 1, 0], [0, -1, 0, 0], [0, 0, 0, 1]], dtype=np.float32)
rendered = 0
for d in sorted(Path(GSO_DIR).iterdir()):
    if not d.is_dir():
        continue
    mcands = list((d/"meshes").glob("*.obj")) + list((d/"meshes").glob("*.ply"))
    if not mcands or (d/"images"/".upright_v4").exists():
        continue
    try:
        tm = trimesh.load(str(mcands[0]), force="mesh")
        if tm is None or len(tm.faces) == 0:
            continue
        tm.apply_transform(rx_gso)
        tm.visual = bake_texture_to_vertices(tm, mesh_path=mcands[0])
        pts = np.array(tm.vertices, dtype=np.float32)
        c = 0.5 * (pts.min(0) + pts.max(0))
        s = np.percentile(np.linalg.norm(pts - c, axis=1), 99.5)
        tm.apply_translation(-c); tm.apply_scale(1.0/(s+1e-8))
        (d/"images").mkdir(parents=True, exist_ok=True)
        # Save 000.png with exact hardware depth-buffer RGBA alpha mask!
        PILImage.fromarray(render_cam(tm, 20.0, 15.0, 512, return_rgba=True), mode="RGBA").save(d/"images"/"000.png")
        PILImage.fromarray(render_cam(tm, 65.0, 15.0, 256, return_rgba=False)).save(d/"images"/"001.png")
        (d/"images"/".upright_v4").write_text("ok")
        rendered += 1
    except Exception as e:
        print("render failed:", d.name, str(e)[:100])

samples = load_gso_samples(GSO_DIR, num_samples=NUM_SAMPLES)
print(f"GSO ready: rendered {rendered} upright RGBA views | loaded {len(samples)} canonical samples ✅")

In [ ]:
# --- SPAR3D (Point-E 512-Pt Diffusion + Triplane DMTet + PBR, ~2.0 s/object on T4) ---
# Wrapped in top-level try/except so Papermill (allow_errors=False) never aborts the kernel!
import os, sys, gc, datetime, shutil, subprocess, types, traceback
from pathlib import Path
import packaging
import torch

try:
    os.environ.setdefault("PYOPENGL_PLATFORM", "egl")
    os.environ["SPAR3D_LOW_VRAM"] = "0"
    os.environ["CC"] = "gcc"
    os.environ["CXX"] = "g++"
    os.environ["TORCH_CUDA_ARCH_LIST"] = "7.5"
    os.environ["MAX_JOBS"] = "4"

    if "CODE_DIR" not in globals():
        CODE_DIR = "/kaggle/working/image-to-3d-benchmark"
    if str(CODE_DIR) not in sys.path:
        sys.path.insert(0, str(CODE_DIR))

    from src.engines.spar3d_engine import Spar3dEngine, _install_spar3d_compat_shims
    from src.evaluation.benchmark_runner import BenchmarkRunner

    _install_spar3d_compat_shims()

    # 1. Install Python dependencies FIRST (including ftfy, regex, loralib) BEFORE importing clip/alpha_clip!
    !pip install -q -c /tmp/constraints.txt einops jaxtyping omegaconf loralib ftfy regex timm kornia

    # 2. Clone CLIP & AlphaCLIP, patch pkg_resources, and add to sys.path
    for _repo_url, _dest in [
        ("https://github.com/openai/CLIP.git", "/kaggle/working/CLIP"),
        ("https://github.com/SunzeY/AlphaCLIP.git", "/kaggle/working/AlphaCLIP"),
    ]:
        if not os.path.isdir(os.path.join(_dest, ".git")):
            shutil.rmtree(_dest, ignore_errors=True)
            subprocess.run(["git", "clone", "--depth", "1", _repo_url, _dest], check=True)
        for _pyf in Path(_dest).rglob("*.py"):
            try:
                _t = _pyf.read_text()
                if "from pkg_resources import packaging" in _t:
                    _pyf.write_text(_t.replace("from pkg_resources import packaging", "import packaging"))
            except Exception:
                pass
        if _dest not in sys.path:
            sys.path.insert(0, _dest)

    import clip, alpha_clip
    print("[SPAR3D] clip and alpha_clip imported cleanly ✅")

    # 3. Clone Stability-AI/stable-point-aware-3d and build sm_75 CUDA extensions
    SPAR3D_DIR = "/kaggle/working/stable-point-aware-3d"
    if not os.path.isdir(os.path.join(SPAR3D_DIR, ".git")):
        shutil.rmtree(SPAR3D_DIR, ignore_errors=True)
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/Stability-AI/stable-point-aware-3d.git", SPAR3D_DIR],
            check=True,
        )
    if SPAR3D_DIR not in sys.path:
        sys.path.insert(0, SPAR3D_DIR)

    try:
        import nvdiffrast.torch  # noqa: F401
    except ImportError:
        !pip install -q -c /tmp/constraints.txt --no-build-isolation git+https://github.com/NVlabs/nvdiffrast.git

    print("[SPAR3D] Building texture_baker and uv_unwrapper CUDA extensions (sm_75) …")
    _tb = subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "--no-build-isolation", "--no-deps", f"{SPAR3D_DIR}/texture_baker"],
        capture_output=True, text=True,
    )
    _uw = subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "--no-build-isolation", "--no-deps", f"{SPAR3D_DIR}/uv_unwrapper"],
        capture_output=True, text=True,
    )
    print(f"[SPAR3D] CUDA extensions: texture_baker={_tb.returncode}, uv_unwrapper={_uw.returncode}")

    sp_engine = Spar3dEngine(device="cuda", texture_resolution=1024, low_vram_mode=False)
    sp_engine.load_model()
    runner = BenchmarkRunner(
        dataset_path=GSO_DIR,
        engines=[sp_engine],
        num_samples=NUM_SAMPLES,
        output_dir="/kaggle/working/outputs/spar3d",
    )
    try:
        print(f"[SPAR3D] {datetime.datetime.now().strftime('%H:%M:%S')} Starting SPAR3D run...")
        runner.run()
        print(f"[SPAR3D] {datetime.datetime.now().strftime('%H:%M:%S')} SPAR3D COMPLETE ✅")
        runner.save_results("/kaggle/working/outputs/spar3d_results.json")
        display(runner.get_summary_df())
    finally:
        del runner, sp_engine
        gc.collect()
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()
except Exception as e:
    print(f"[SPAR3D] Failed with {type(e).__name__}: {e}")
    traceback.print_exc()

In [ ]:
# --- TRELLIS (Sparse Structure Flow + SLAT Rectified Flow -> 256^3 FlexiCubes) ---
# Pre-seeds verified 10-object batch2_v16 TRELLIS results when RUN_TRELLIS=False
# (flip RUN_TRELLIS=True when running 100 objects).
import os, sys, gc, glob, json, shutil, subprocess, datetime, types, traceback
from pathlib import Path
import torch

RUN_TRELLIS = True

try:
    os.environ.setdefault("PYOPENGL_PLATFORM", "egl")
    os.environ["ATTN_BACKEND"] = "xformers"
    os.environ["SPCONV_ALGO"] = "native"
    os.environ["SPCONV_DISABLE_JIT"] = "1"
    os.environ["TORCH_CUDA_ARCH_LIST"] = "7.5"
    os.environ["MAX_JOBS"] = "4"

    _trellis_out = Path("/kaggle/working/outputs/trellis_results.json")
    _trellis_out.parent.mkdir(parents=True, exist_ok=True)
    _VERIFIED_TRELLIS_V16 = '[\n  {\n    "engine": "TRELLIS",\n    "sample_id": "3D_Dollhouse_Happy_Brother",\n    "latency_s": 20.761,\n    "vram_peak_gb": 6.77,\n    "chamfer_distance": 0.12171495789327799,\n    "f_score_0.1": 0.8121203709390162,\n    "f_score_0.2": 0.9856492669811799,\n    "f_score_0.5": 1.0,\n    "psnr": 21.728962918476252,\n    "lpips": 0.18720743060112,\n    "metadata": {\n      "engine": "TRELLIS",\n      "output_format": "mesh",\n      "latency_s": 18.767,\n      "slat_steps": 8,\n      "sparse_steps": 8,\n      "vertex_count": 207402,\n      "face_count": 414816,\n      "has_vertex_colors": true\n    }\n  },\n  {\n    "engine": "TRELLIS",\n    "sample_id": "3D_Dollhouse_Sofa",\n    "latency_s": 24.058,\n    "vram_peak_gb": 7.84,\n    "chamfer_distance": 0.21900848642634785,\n    "f_score_0.1": 0.6047111366811869,\n    "f_score_0.2": 0.8208808071044523,\n    "f_score_0.5": 0.9937106918238994,\n    "psnr": 17.246598021087266,\n    "lpips": 0.3317089378833771,\n    "metadata": {\n      "engine": "TRELLIS",\n      "output_format": "mesh",\n      "latency_s": 22.071,\n      "slat_steps": 8,\n      "sparse_steps": 8,\n      "vertex_count": 344864,\n      "face_count": 689700,\n      "has_vertex_colors": true\n    }\n  },\n  {\n    "engine": "TRELLIS",\n    "sample_id": "ASICS_GELAce_Pro_Pearl_WhitePink",\n    "latency_s": 18.186,\n    "vram_peak_gb": 6.82,\n    "chamfer_distance": 0.07208778636439883,\n    "f_score_0.1": 0.9492622409258286,\n    "f_score_0.2": 0.998598037252153,\n    "f_score_0.5": 1.0,\n    "psnr": 15.96991120125452,\n    "lpips": 0.16631892323493958,\n    "metadata": {\n      "engine": "TRELLIS",\n      "output_format": "mesh",\n      "latency_s": 16.225,\n      "slat_steps": 8,\n      "sparse_steps": 8,\n      "vertex_count": 236342,\n      "face_count": 472740,\n      "has_vertex_colors": true\n    }\n  },\n  {\n    "engine": "TRELLIS",\n    "sample_id": "ASICS_GELLinksmaster_WhiteCoffeeSand",\n    "latency_s": 20.1105,\n    "vram_peak_gb": 7.21,\n    "chamfer_distance": 0.12532655950055827,\n    "f_score_0.1": 0.7734424495566122,\n    "f_score_0.2": 0.9830513146518843,\n    "f_score_0.5": 1.0,\n    "psnr": 13.860031336634504,\n    "lpips": 0.22615548968315125,\n    "metadata": {\n      "engine": "TRELLIS",\n      "output_format": "mesh",\n      "latency_s": 18.152,\n      "slat_steps": 8,\n      "sparse_steps": 8,\n      "vertex_count": 282160,\n      "face_count": 564328,\n      "has_vertex_colors": true\n    }\n  },\n  {\n    "engine": "TRELLIS",\n    "sample_id": "ASICS_GELResolution_5_Flash_YellowBlackSilver",\n    "latency_s": 18.9246,\n    "vram_peak_gb": 6.98,\n    "chamfer_distance": 0.07073369024091063,\n    "f_score_0.1": 0.9405054593733404,\n    "f_score_0.2": 1.0,\n    "f_score_0.5": 1.0,\n    "psnr": 19.921394007749974,\n    "lpips": 0.17921221256256104,\n    "metadata": {\n      "engine": "TRELLIS",\n      "output_format": "mesh",\n      "latency_s": 16.937,\n      "slat_steps": 8,\n      "sparse_steps": 8,\n      "vertex_count": 252062,\n      "face_count": 504160,\n      "has_vertex_colors": true\n    }\n  },\n  {\n    "engine": "TRELLIS",\n    "sample_id": "ASICS_HyperRocketgirl_SP_5_WhiteMalibu_BlueBlack",\n    "latency_s": 17.2795,\n    "vram_peak_gb": 6.58,\n    "chamfer_distance": 0.09167234413094782,\n    "f_score_0.1": 0.9049065128601391,\n    "f_score_0.2": 1.0,\n    "f_score_0.5": 1.0,\n    "psnr": 21.481688544669417,\n    "lpips": 0.15159361064434052,\n    "metadata": {\n      "engine": "TRELLIS",\n      "output_format": "mesh",\n      "latency_s": 15.296,\n      "slat_steps": 8,\n      "sparse_steps": 8,\n      "vertex_count": 234518,\n      "face_count": 469084,\n      "has_vertex_colors": true\n    }\n  },\n  {\n    "engine": "TRELLIS",\n    "sample_id": "Android_Figure_Panda",\n    "latency_s": 19.6132,\n    "vram_peak_gb": 7.11,\n    "chamfer_distance": 0.3074675841197856,\n    "f_score_0.1": 0.3338502383359815,\n    "f_score_0.2": 0.7232517708548243,\n    "f_score_0.5": 0.9985481748535375,\n    "psnr": 15.240088092669431,\n    "lpips": 0.23943503201007843,\n    "metadata": {\n      "engine": "TRELLIS",\n      "output_format": "mesh",\n      "latency_s": 17.636,\n      "slat_steps": 8,\n      "sparse_steps": 8,\n      "vertex_count": 283504,\n      "face_count": 567008,\n      "has_vertex_colors": true\n    }\n  },\n  {\n    "engine": "TRELLIS",\n    "sample_id": "Animal_Planet_Foam_2Headed_Dragon",\n    "latency_s": 15.8529,\n    "vram_peak_gb": 6.36,\n    "chamfer_distance": 0.09316202500414592,\n    "f_score_0.1": 0.9111466388629753,\n    "f_score_0.2": 0.9912253909008373,\n    "f_score_0.5": 1.0,\n    "psnr": 19.598659310158347,\n    "lpips": 0.13295115530490875,\n    "metadata": {\n      "engine": "TRELLIS",\n      "output_format": "mesh",\n      "latency_s": 13.902,\n      "slat_steps": 8,\n      "sparse_steps": 8,\n      "vertex_count": 98952,\n      "face_count": 197860,\n      "has_vertex_colors": true\n    }\n  },\n  {\n    "engine": "TRELLIS",\n    "sample_id": "Azure_Snake_Tieks_Leather_Snake_Print_Ballet_Flats",\n    "latency_s": 17.5507,\n    "vram_peak_gb": 6.71,\n    "chamfer_distance": 0.08528980305880059,\n    "f_score_0.1": 0.9195152846501006,\n    "f_score_0.2": 0.9928999446094969,\n    "f_score_0.5": 1.0,\n    "psnr": 20.491577864378673,\n    "lpips": 0.18710175156593323,\n    "metadata": {\n      "engine": "TRELLIS",\n      "output_format": "mesh",\n      "latency_s": 15.587,\n      "slat_steps": 8,\n      "sparse_steps": 8,\n      "vertex_count": 191530,\n      "face_count": 383060,\n      "has_vertex_colors": true\n    }\n  },\n  {\n    "engine": "TRELLIS",\n    "sample_id": "Bradshaw_International_11642_7_Qt_MP_Plastic_Bowl",\n    "latency_s": 24.6874,\n    "vram_peak_gb": 7.61,\n    "chamfer_distance": 0.3442886445333595,\n    "f_score_0.1": 0.33660273348519365,\n    "f_score_0.2": 0.6079198094481936,\n    "f_score_0.5": 0.982706002034588,\n    "psnr": 11.940202765309213,\n    "lpips": 0.4066570997238159,\n    "metadata": {\n      "engine": "TRELLIS",\n      "output_format": "mesh",\n      "latency_s": 22.626,\n      "slat_steps": 8,\n      "sparse_steps": 8,\n      "vertex_count": 522962,\n      "face_count": 1045850,\n      "has_vertex_colors": true\n    }\n  }\n]'

    if not RUN_TRELLIS and NUM_SAMPLES <= 10:
        _trellis_out.write_text(_VERIFIED_TRELLIS_V16)
        print("[TRELLIS] Wrote verified 10-object batch2_v16 results -> /kaggle/working/outputs/trellis_results.json ✅")
    else:
        !pip install -q -c /tmp/constraints.txt "pybind11==2.13.6" cumm-cu126 spconv-cu126
        _c8incs = sorted(glob.glob("/usr/local/lib/python3*/dist-packages/cumm/include")) + \
                  sorted(glob.glob("/opt/conda/lib/python3*/site-packages/cumm/include"))
        if _c8incs:
            os.environ["CPATH"] = _c8incs[0] + (os.pathsep + os.environ["CPATH"] if os.environ.get("CPATH") else "")

        try:
            import xformers  # noqa: F401
        except ImportError:
            os.system("pip install -q -c /tmp/constraints.txt --no-deps xformers")

        !pip install -q -c /tmp/constraints.txt pytorch-lightning omegaconf easydict "imageio[ffmpeg]" timm kornia open3d
        try:
            import nvdiffrast.torch  # noqa: F401
        except ImportError:
            !pip install -q -c /tmp/constraints.txt --no-build-isolation git+https://github.com/NVlabs/nvdiffrast.git

        def _ensure_checkout(repo_url, dest, pin):
            if os.path.isdir(dest) and not os.path.isdir(os.path.join(dest, ".git")):
                shutil.rmtree(dest, ignore_errors=True)
            if not os.path.isdir(os.path.join(dest, ".git")):
                subprocess.run(["git", "clone", "--depth", "1", "--recurse-submodules", repo_url, dest], check=True)
            subprocess.run(["git", "-C", dest, "checkout", "-f", pin], capture_output=True)

        _ensure_checkout("https://github.com/microsoft/TRELLIS.git", "/kaggle/working/TRELLIS", "442aa1e1afb9014e80681d3bf604e8d728a86ee7")
        if "/kaggle/working/TRELLIS" not in sys.path:
            sys.path.insert(0, "/kaggle/working/TRELLIS")

        # Patch TRELLIS base.py: surface swallowed primary-load errors instead of
        # silently falling back to a bogus HF repo id (bare `v` -> 401).
        _base_py = "/kaggle/working/TRELLIS/trellis/pipelines/base.py"
        _base_src = open(_base_py).read()
        _old = "            except:\n                _models[k] = models.from_pretrained(v)"
        _new = (
            "            except Exception as _primary_err:\n"
            "                print(f\"[TRELLIS-loader] {k}: primary load failed: {type(_primary_err).__name__}: {_primary_err}\")\n"
            "                if os.path.isdir(path):\n"
            "                    raise\n"
            "                _models[k] = models.from_pretrained(v)"
        )
        if _old in _base_src:
            open(_base_py, "w").write(_base_src.replace(_old, _new))
            print("[TRELLIS] Patched base.py to surface primary-load errors ✅")
        elif "[TRELLIS-loader]" in _base_src:
            print("[TRELLIS] base.py already patched ✅")
        else:
            print("[TRELLIS] WARNING: base.py patch anchor not found (upstream changed?)")

        if "kaolin" not in sys.modules:
            _kaolin = types.ModuleType("kaolin")
            _kaolin_utils = types.ModuleType("kaolin.utils")
            _kaolin_testing = types.ModuleType("kaolin.utils.testing")
            _kaolin_testing.check_tensor = lambda tensor, shape=None, throw=True, **_kw: True
            _kaolin.utils = _kaolin_utils
            _kaolin.utils.testing = _kaolin_testing
            sys.modules["kaolin"] = _kaolin
            sys.modules["kaolin.utils"] = _kaolin_utils
            sys.modules["kaolin.utils.testing"] = _kaolin_testing

        _ensure_checkout("https://github.com/JeffreyXiang/diffoctreerast.git", "/tmp/extensions/diffoctreerast", "b09c20b84ec3aace4729e6e18a613112320eca3a")
        !pip install -q --no-build-isolation /tmp/extensions/diffoctreerast
        !pip install -q git+https://github.com/EasternJournalist/utils3d.git@9a4eb15e4021b67b12c460c7057d642626897ec8

        from src.engines.trellis_engine import TrellisEngine
        from src.evaluation.benchmark_runner import BenchmarkRunner

        tr_engine2 = TrellisEngine(device="cuda", output_format="mesh", slat_steps=8, sparse_steps=8, staged_offload=True)
        tr_engine2.load_model()
        runner = BenchmarkRunner(dataset_path=GSO_DIR, engines=[tr_engine2], num_samples=NUM_SAMPLES, output_dir="/kaggle/working/outputs/trellis")
        try:
            runner.run()
            runner.save_results("/kaggle/working/outputs/trellis_results.json")
            display(runner.get_summary_df())
        finally:
            del runner, tr_engine2
            gc.collect()
            torch.cuda.empty_cache()
            torch.cuda.ipc_collect()
except Exception as e:
    print(f"[TRELLIS] Error: {type(e).__name__}: {e}")
    traceback.print_exc()

In [ ]:
import glob, json
import pandas as pd
import matplotlib.pyplot as plt

rows = []
for p in sorted(glob.glob("/kaggle/working/outputs/*_results.json")):
    try:
        rows += json.load(open(p))
    except Exception as e:
        print("skip", p, str(e)[:80])

df = pd.DataFrame(rows)
df.to_csv("/kaggle/working/outputs/unified_results.csv", index=False)

num_cols = ["latency_s", "vram_peak_gb", "chamfer_distance", "f_score_0.1", "f_score_0.2", "f_score_0.5", "psnr", "lpips"]
cols = [c for c in num_cols if c in df.columns]
if cols and not df.empty:
    summary = df.groupby("engine")[cols].agg(["mean", "std"]).round(4)
    print("=== UNIFIED BENCHMARK RESULTS (mean +/- std) ===")
    print(summary.to_string())
    with open("/kaggle/working/outputs/unified_table.tex", "w") as f:
        f.write(summary.reset_index().to_latex(index=False, na_rep="--"))

    colors = {"TripoSR": "#1f77b4", "LGM": "#ff7f0e", "TRELLIS": "#2ca02c", "SPAR3D": "#9467bd"}
    means = df.groupby("engine")[cols].mean()
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    for eng, row in means.iterrows():
        c = colors.get(eng, "grey")
        if "f_score_0.1" in row and not pd.isna(row.get("f_score_0.1")):
            axes[0].scatter(row["latency_s"], row["f_score_0.1"], s=180, c=c, label=eng, zorder=5)
        if "chamfer_distance" in row and not pd.isna(row.get("chamfer_distance")):
            axes[1].scatter(row["latency_s"], row["chamfer_distance"], s=180, c=c, label=eng, zorder=5)
    axes[0].set_xlabel("Latency (s)"); axes[0].set_ylabel("F-Score @ 0.1"); axes[0].legend(); axes[0].grid(alpha=0.3)
    axes[1].set_xlabel("Latency (s)"); axes[1].set_ylabel("Chamfer Distance"); axes[1].legend(); axes[1].grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig("/kaggle/working/outputs/pareto_frontier.png", dpi=150)
    print("saved: unified_results.csv, unified_table.tex, pareto_frontier.png ✅")

In [ ]:
import shutil, os
from pathlib import Path

WORK = Path("/kaggle/working")
KEEP = {"outputs"}
if WORK.exists():
    for item in sorted(WORK.iterdir()):
        if item.name in KEEP:
            continue
        try:
            if item.is_dir():
                shutil.rmtree(item)
            else:
                item.unlink()
        except Exception:
            pass
    print("saved outputs:", sorted(p.name for p in (WORK / "outputs").iterdir()))